# Finite-Rank Operator Construction and Eigenpair Verification

This notebook constructs a finite-rank Galerkin discretization of the operator
and verifies approximate eigenpairs of the resulting generalized eigenvalue problem.

The main objectives are:

- Construct finite-dimensional matrices representing the operator.
- Verify approximate eigenpairs through residual evaluation.
- Evaluate the operator applied to the eigenfunction for further verification.

Specifically, we verify approximate eigenpairs for the **vector divergence-free Laplacian**
through a finite-dimensional Galerkin discretization.

---

## 1. Basis / eigenfunction evaluation

We first evaluate the divergence-free basis functions (and the associated eigenfunction representation)
used to discretize the operator.

---

## 2. Assemble the generalized eigenvalue problem

We assemble two matrices

$$
A_{ij}=\langle \phi_i, Q\phi_j\rangle,
\qquad
B_{ij}=\langle \phi_i, \phi_j\rangle
$$

where $\{\phi_i\}$ are the basis functions and $\langle\cdot,\cdot\rangle$ denotes the chosen inner product.

---

## 3. Residual verification for approximate eigenpairs

Given approximate eigenpairs $(\lambda, x)$ of

$$
A x = \lambda B x,
$$

we verify that they satisfy the eigenvalue problem up to a small residual

$$
r := A x - \lambda B x,
$$

and bound $\|r\|$ in the norm used in this notebook.

---

## 4. Pointwise evaluation for additional checks

Finally, we evaluate $Q\psi$ on mesh grid points for further independent verification.

In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/julia code/residual_verification`


.Interval{Float64}
Number of threads = 56


In [2]:
# Load parameters defining the divergence-free basis functions.
# Y_paras and Z_paras encode the Bessel zero data used to construct radial basis families.
Y_paras, Z_paras = load_besselj_zeros("../data/bessel_zeros.mat"; refine=!(dtype==Float64))
l = max(length(Y_paras), length(Z_paras))

# Load the operator coefficient/function Q (stored in frequency form).
# This Q will be used in the bilinear form <φ_i, Q φ_j>.
Q = load_grad_U("../data/approx_Q.mat")

# Discretization / truncation parameters for assembly and quadrature.
N0 = 18000 # radial grid resolution (before extension)
M0, M1 = size(Q["A11"].freq_func) .* 2 # spectral resolution inferred from Q
R = dt(32) # physical truncation radius (r ∈ [0, R])
eta = dt("0.005") # regularization / weighting parameter used in Laplacian matrix assembly
k = 8 # stencil half-width for finite differences (ghost points)
;

In [3]:
# Extend the radial grid with k ghost points on both sides.
# This supports finite-difference stencils / derivative bounds used in later steps.
r_ext = dt.(-k:N0+k) ./ dt(N0)
r = r_ext[k+1:end-k]; # interior grid on [0, 1] (scaled radius)
r_Q = r_ext .* (Pi/dt(2)) # mapped radius used inside Q integration routines

# Integrate <φ_i, Q φ_j> in the θ direction first.
# Output is a vector-valued function of r (for each mode pair), with rigorous error bounds.
result_odd, result_even = integrate_θ(Q, r_Q, l, M1);

In [4]:
# Optional: cache θ-integrated results to disk for reuse.
# This avoids recomputing the expensive integrate_θ step.
save_matrix("../data/integrate_θ_result.mat", Dict(
    "result_even" => result_even, "result_odd" => result_odd))
    
# Uncomment to load cached results:
# result_even, result_odd = read_matrix("../data/integrate_θ_result.mat", 
#     ["result_even", "result_odd"]);

In [5]:
# Evaluate radial components Ur of the divergence-free basis on the interior grid r.
# Different families (Y/X/Z) correspond to different basis constructions.
Ur_Y_lst = eval_Ur(r, Y_paras, "Y")
Ur_X_lst = eval_Ur(r, Y_paras, "X")
Ur_Z_lst = eval_Ur(r, Z_paras, "Z")

# Compute rigorous bounds for Ur and its derivatives up to order k,
# used to control quadrature / truncation errors in matrix assembly.
Y_bound_lst, X_bound_lst, Z_bound_lst = Ur_bound_lst(Y_paras, Z_paras, k)

# Assemble the mass matrix B (inner products <φ_i, φ_j>) and the Laplacian-related structure,
# together with rigorous error bounds for the assembly.
B_even, B_odd, B_even_err, B_odd_err = get_lap_matrix(Ur_Y_lst, Ur_X_lst, Ur_Z_lst, 
    Y_bound_lst, X_bound_lst, Z_bound_lst, Y_paras, Z_paras, r, R, eta)

# Assemble the operator matrix A (inner products <φ_i, Q φ_j>),
# using θ-integrated results and rigorous quadrature/error bounds.
A_even, A_odd, A_even_err, A_odd_err = integrate_r([result_even, result_odd], 
    Ur_Y_lst, Ur_X_lst, Ur_Z_lst, Y_bound_lst, X_bound_lst, Z_bound_lst, r, M0);

Any[IntervalArithmetic.Interval{Float64}[[0.333333, 0.333333]_com, [0.838442, 0.838442]_trv, [2.94855, 2.94855]_trv, [6.0931, 6.0931]_trv, [10.283, 10.283]_trv, [15.5195, 15.5195]_trv, [21.803, 21.803]_trv, [29.1336, 29.1336]_trv, [37.5113, 37.5113]_trv, [46.9361, 46.9361]_trv, [57.4082, 57.4082]_trv, [68.9274, 68.9274]_trv, [81.4938, 81.4938]_trv, [95.1074, 95.1074]_trv, [109.768, 109.768]_trv, [125.476, 125.476]_trv, [142.231, 142.231]_trv, [160.034, 160.034]_trv, [178.883, 178.883]_trv, [198.78, 198.78]_trv, [219.724, 219.724]_trv, [241.715, 241.715]_trv, [264.754, 264.754]_trv, [288.839, 288.839]_trv, [313.972, 313.972]_trv, [340.152, 340.152]_trv, [367.379, 367.379]_trv, [395.653, 395.653]_trv, [424.975, 424.975]_trv, [455.343, 455.343]_trv, [486.759, 486.759]_trv, [519.223, 519.223]_trv], IntervalArithmetic.Interval{Float64}[[0.0130613, 0.0130613]_trv, [0.162924, 0.162924]_trv, [0.661925, 0.661925]_trv, [1.38743, 1.38743]_trv, [2.32658, 2.32658]_trv, [3.47673, 3.47673]_trv, [4.83

In [6]:
# If we are in interval arithmetic mode (dtype != Float64),
# incorporate the rigorous error bounds into A and B as interval radii.
if dtype != Float64
    A_even = add_error(A_even, A_even_err)
    A_odd = add_error(A_odd, A_odd_err)
    B_even = add_error(B_even, B_even_err)
    B_odd = add_error(B_odd, B_odd_err);
end

# Rescale by R^3 because the radial variable was scaled from [0, R] to [0, 1]
# during quadrature / basis evaluation.
A_even .*= R^3;
A_odd .*= R^3;
B_even .*= R^3;
B_odd .*= R^3;

In [7]:
# Optional: cache assembled Galerkin matrices A and B for reuse.
save_matrix("../data/psi_eig_mat_AB.mat", Dict(
    "A_even" => A_even, "A_odd" => A_odd,
    "B_even" => B_even, "B_odd" => B_odd))

# Uncomment to load cached matrices:
# A_even, A_odd, B_even, B_odd = read_matrix(
#     "../data/psi_eig_mat_AB.mat", 
#     ["A_even", "A_odd", "B_even", "B_odd"]
# );

## Eigenpair verification and computation of Qψ

We load approximate eigenpairs $(\lambda, V)$ of the generalized eigenvalue problems

$$
A V = B V \Lambda^{-1},
$$

and compute the following diagnostic quantities:

- Orthogonality error:
  $$
  V^T B V - I
  $$

- Consistency error:
  $$
  I - \Lambda^{1/2} V^T A V \Lambda^{1/2}
  $$

- Residual matrix:
  $$
  A V \Lambda - B V
  $$

The norms of these matrices are printed to quantify how accurately the eigenpairs satisfy the Galerkin eigenvalue problem.

Finally, we reconstruct the eigenfunction $\psi$ and evaluate

$$
Q\psi
$$

on a global mesh grid. The resulting arrays `rhs_even` and `rhs_odd` are saved for further verification.

In [14]:
# Load precomputed approximate eigenpairs for the generalized eigenproblems:
#   A_even x = λ B_even x,   A_odd x = λ B_odd x.
eig_val_even, eig_vec_even, eig_val_odd, eig_vec_odd = read_matrix("../data/eig.mat", 
    ["eig_val_even", "eig_vec_even", "eig_val_odd", "eig_vec_odd"]);

In [15]:
# Verify the loaded approximate eigenpairs using a residual-based test.
# 'thr' sets a tolerance / threshold used inside verify_general_eig.
thr = dt("0.456")
println(verify_general_eig(A_even, B_even, eig_vec_even, eig_val_even[:], thr))
println(verify_general_eig(A_odd, B_odd, eig_vec_odd, eig_val_odd[:], thr))

true
true


In [16]:
using LinearAlgebra

# Verify the residuals of the eigenvalue problems and the orthogonality
GE_even = eig_vec_even' * B_even * eig_vec_even
GQ_even = eig_vec_even' * A_even * eig_vec_even
GE_odd = eig_vec_odd' * B_odd * eig_vec_odd
GQ_odd = eig_vec_odd' * A_odd * eig_vec_odd

Lambda_even = diagm(inv.(eig_val_even[:]))
Lambda_odd = diagm(inv.(eig_val_odd[:]))
I_even = Diagonal(ones(dtype, size(GE_even, 1)))
I_odd = Diagonal(ones(dtype, size(GE_odd, 1)))

# All these errors should be close to zero
eQ_even = I_even - sqrt.(Lambda_even) * GQ_even * sqrt.(Lambda_even)
eQ_odd = I_odd - sqrt.(Lambda_odd) * GQ_odd * sqrt.(Lambda_odd);
eE_even = I_even - GE_even
eE_odd = I_odd - GE_odd;
eD_even = (A_even * eig_vec_even * Lambda_even .- B_even * eig_vec_even)
eD_odd = (A_odd * eig_vec_odd * Lambda_odd .- B_odd * eig_vec_odd);

In [17]:
# Report norms of the diagnostic errors used in the verification step.
# eQ and eE measure (normalized) eigen-consistency and B-orthogonality.
# eD measures the generalized eigen-residual in matrix form.
println("eQ_even norm: ", norm(vec(eQ_even)))
println("eQ_odd norm: ", norm(vec(eQ_odd)))
println("eD_even norm: ", norm(vec(eD_even)) / sqrt(one(dtype) - norm(vec(eD_even))))
println("eD_odd norm: ", norm(vec(eD_odd)) / sqrt(one(dtype) - norm(vec(eD_odd))))

eQ_even norm: [1.51455e-9, 2.22922e-8]_com
eQ_odd norm: [1.91354e-11, 2.75918e-8]_com
eD_even norm: [9.73822e-8, 1.47482e-5]_com
eD_odd norm: [1.2872e-9, 5.28861e-5]_com


In [18]:
# Set up a high-resolution global mesh for pointwise evaluation checks.
# We map between (β, θ) and the scaled radius r via:
#   β ∈ [0, atan(R/scl_fac)],  r = tan(β)*scl_fac / R.
N0_glb, N1_glb = 12000, 6000
k = 10

# Load scaling factor used in the β ↔ r transformation.
scl_fac = dt(read_matrix("../data/UP.mat", ["scl_fac"]))
Beta = atan(R / scl_fac)

# Extend grids by k ghost points for derivative stencils / safe evaluation.
β_glb = dt.(-k:(N0_glb+k)) / dt(N0_glb) * Beta
θ_glb = dt.(-k:(N1_glb+k)) / dt(N1_glb) * (Pi/dt(2))

# Convert to scaled radial coordinate r ∈ [0, 1] (with extension).
r_glb = tan.(β_glb) * scl_fac / R;

In [19]:
# Evaluate Q ψ on the global mesh for an additional pointwise verification step.
# The output rhs_even / rhs_odd corresponds to the operator applied to the eigenfunction
# in the chosen representation.
rhs_even, rhs_odd = eval_Q_eig_function(eig_vec_odd, eig_vec_even, Y_paras, Z_paras, r_glb, θ_glb, Q);

# Apply the geometric weight tan(β)/cos(β) used in the target formulation.
weight_glb = tan.(β_glb) ./ cos.(β_glb)
rhs_even .*= weight_glb
rhs_odd .*= weight_glb;

# Save rhs for downstream verification / plotting.
save_matrix("../data/rhs.mat", Dict("rhs_even" => rhs_even, "rhs_odd" => rhs_odd))